# Check Code PFROMZ validation lab — V0.1

Validate the browser candidate's **normal percentile** contract with an independent Python `math.erf` oracle. `PFROMZ` is not a hypothesis-test p-value. Passing is evidence, not statistical approval or complete desktop parity.

In [ ]:
import math
from pyodide.http import pyfetch
response = await pyfetch('../../validation-fixtures/check-code-pfromz-v0.1.json')
response.raise_for_status()
fixture = await response.json()
fixture['operation'], fixture['method']

In [ ]:
def independent_normal_percentile(z):
    return 100.0 * 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def retained_display_profile(value):
    rounded = round(value, 2)  # Python and retained C# use midpoint-to-even.
    return 99.99 if rounded >= 99.9999 else (-99.99 if rounded <= -99.9999 else rounded)

rows = []
for case in fixture['cases']:
    raw = independent_normal_percentile(case['z'])
    oracle = retained_display_profile(raw)
    assert abs(oracle - case['expected']) <= fixture['tolerance']
    rows.append({'z': case['z'], 'rawPercentile': raw, 'expectedCandidate': case['expected'], 'oracleProfile': oracle})
rows

In [ ]:
central = [-3.0, -1.96, -1.0, 0.0, 1.0, 1.96, 3.0]
values = [independent_normal_percentile(z) for z in central]
assert all(left < right for left, right in zip(values, values[1:])), 'CDF must be monotone'
for z in [0.0, 1.0, 1.96, 3.0]:
    assert math.isclose(independent_normal_percentile(-z) + independent_normal_percentile(z), 100.0, abs_tol=1e-12)
{'status': 'PASS', 'oracle': 'Python math.erf', 'cases': len(rows), 'quantity': 'normal percentile'}